# House Prices: Advanced Regression Techniques
**Author:** Koketso Raphasha | [Kaggle: Raphasha27](https://kaggle.com/Raphasha27)

**Approach:** Feature engineering, log transform, Ridge/Lasso/GB/XGB ensemble

**Metric:** Root Mean Squared Logarithmic Error (RMSLE)

In [ ]:
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import KFold, cross_val_score
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import Ridge, Lasso, ElasticNet
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from xgboost import XGBRegressor
from sklearn.metrics import mean_squared_log_error

warnings.filterwarnings('ignore')
sns.set_style('darkgrid')
plt.rcParams['figure.figsize'] = (12, 6)

In [ ]:
train = pd.read_csv('/kaggle/input/house-prices-advanced-regression-techniques/train.csv')
test = pd.read_csv('/kaggle/input/house-prices-advanced-regression-techniques/test.csv')
print(f'Train: {train.shape[0]} rows x {train.shape[1]} cols')
print(f'Test:  {test.shape[0]} rows x {test.shape[1]} cols')
print(f'\nMissing:\n{train.isnull().sum().sort_values(ascending=False).head(20)}')

## Exploratory Data Analysis

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
sns.histplot(train['SalePrice'], kde=True, ax=axes[0,0])
axes[0,0].set_title('SalePrice Distribution')
sns.scatterplot(data=train, x='GrLivArea', y='SalePrice', ax=axes[0,1])
axes[0,1].set_title('Above Grade Living Area vs Price')
sns.boxplot(data=train, x='OverallQual', y='SalePrice', ax=axes[0,2])
axes[0,2].set_title('Overall Quality vs Price')
sns.scatterplot(data=train, x='TotalBsmtSF', y='SalePrice', ax=axes[1,0])
axes[1,0].set_title('Total Basement SF vs Price')
sns.scatterplot(data=train, x='YearBuilt', y='SalePrice', ax=axes[1,1])
axes[1,1].set_title('Year Built vs Price')
corr = train.select_dtypes(include=[np.number]).corr()['SalePrice'].abs().sort_values(ascending=False).head(10)
corr.plot(kind='bar', ax=axes[1,2])
axes[1,2].set_title('Top 10 Correlations with SalePrice')
plt.tight_layout()
plt.savefig('eda.png', dpi=100)
plt.show()

## Feature Engineering

In [ ]:
def preprocess(df, is_train=True):
    data = df.copy()
    if is_train:
        data['LogSalePrice'] = np.log1p(data['SalePrice'])
    
    data['TotalSF'] = data['TotalBsmtSF'] + data['1stFlrSF'] + data['2ndFlrSF']
    data['TotalBath'] = data['FullBath'] + 0.5*data['HalfBath'] + data['BsmtFullBath'] + 0.5*data['BsmtHalfBath']
    data['TotalPorchSF'] = data['OpenPorchSF'] + data['EnclosedPorch'] + data['3SsnPorch'] + data['ScreenPorch']
    data['HasPool'] = (data['PoolArea'] > 0).astype(int)
    data['HasFireplace'] = (data['Fireplaces'] > 0).astype(int)
    data['HasGarage'] = (data['GarageArea'] > 0).astype(int)
    data['HasBsmt'] = (data['TotalBsmtSF'] > 0).astype(int)
    data['Age'] = data['YrSold'] - data['YearBuilt']
    data['RemodAge'] = data['YrSold'] - data['YearRemodAdd']
    data['LotFrontage'] = data.groupby('Neighborhood')['LotFrontage'].transform(lambda x: x.fillna(x.median()))
    return data

train_fe = preprocess(train, is_train=True)
test_fe = preprocess(test, is_train=False)

# Log transform skewed numeric features
numeric_feats = train_fe.select_dtypes(include=[np.number]).columns.tolist()
skewed = train_fe[numeric_feats].skew().sort_values(ascending=False)
skewed = skewed[skewed > 0.75]
skewed_cols = skewed.index.tolist()
for col in skewed_cols:
    if col != 'LogSalePrice':
        train_fe[col] = np.log1p(train_fe[col].clip(lower=0))
        test_fe[col] = np.log1p(test_fe[col].clip(lower=0))

# Label encode categoricals
cat_cols = train_fe.select_dtypes(include=['object']).columns.tolist()
for col in cat_cols:
    le = LabelEncoder()
    combined = pd.concat([train_fe[col], test_fe[col]], axis=0).fillna('None').astype(str)
    le.fit(combined)
    train_fe[col] = le.transform(train_fe[col].fillna('None').astype(str))
    test_fe[col] = le.transform(test_fe[col].fillna('None').astype(str))

y = train_fe['LogSalePrice'].values
feature_cols = [c for c in train_fe.columns if c not in ('Id', 'SalePrice', 'LogSalePrice') and c in test_fe.columns]
X = train_fe[feature_cols]
X_test = test_fe[feature_cols]

# Fill remaining NaN
X = X.fillna(X.median())
X_test = X_test.fillna(X.median())

scaler = StandardScaler()
X_s = scaler.fit_transform(X)
X_test_s = scaler.transform(X_test)

print(f'Features: {X.shape[1]}')

## Model Training

In [ ]:
kf = KFold(n_splits=5, shuffle=True, random_state=42)

models = {
    'Ridge': Ridge(alpha=15.0, random_state=42),
    'Lasso': Lasso(alpha=0.001, random_state=42),
    'ElasticNet': ElasticNet(alpha=0.001, l1_ratio=0.5, random_state=42),
    'GB': GradientBoostingRegressor(n_estimators=300, max_depth=3, learning_rate=0.05,
                                     subsample=0.8, random_state=42),
    'XGB': XGBRegressor(n_estimators=500, max_depth=4, learning_rate=0.03,
                         subsample=0.8, colsample_bytree=0.8, random_state=42),
}

for name, model in models.items():
    scores = -cross_val_score(model, X_s, y, cv=kf, scoring='neg_root_mean_squared_error')
    print(f'{name}: CV RMSE {scores.mean():.4f} (+/- {scores.std()*2:.4f})')

# Ensemble predictions
test_preds = np.zeros((X_test.shape[0], len(models)))
for i, (name, model) in enumerate(models.items()):
    model.fit(X_s, y)
    test_preds[:, i] = model.predict(X_test_s)

final_preds = np.expm1(test_preds.mean(axis=1))
print(f'\nPredictions: min={final_preds.min():.0f}, max={final_preds.max():.0f}, mean={final_preds.mean():.0f}')

## Submission

In [ ]:
sub = pd.DataFrame({'Id': test['Id'], 'SalePrice': final_preds})
sub.to_csv('submission.csv', index=False)
print('Ready: submission.csv')
sub.head()